# Dependency

- vedo: 2024.5.2

# Common Libraries

In [1]:
import os, sys, glob
import numpy as np
from pathlib import Path
import matplotlib.pylab as plt
from skimage import io
from matplotlib.colors import to_hex
from vedo import settings
settings.default_backend = "vtk"

# Paths

In [ ]:
# afni path
abin_path = "/Users/seojin/abin"

# Directory path for saving cluster result
cluster_dir_path = "/Users/seojin/Desktop/DP_visualization/brain_rois/clusters"

# Base brain mask - LPS+ coords(mni, spm)
base_brain_nii_path = "/Users/seojin/Desktop/MRI_visualization/Vedo_visualization/group_mask.nii.gz"

# Brain roi path
brain_roi_path = "/Users/seojin/Desktop/DP_visualization/brain_rois"

# Custom Libraries

In [ ]:
# Custom Libraries
from afni_extension import set_afni_abin
from brain_vis import show_clusterize_brain
from sj_datastructure import sort_usingRef
from color_util import hex_to_rgb

set_afni_abin(abin_path)

# Parameters

In [10]:
# Plotting method
cluster_plot_style = "mesh" # mesh, point

# Query atlas based on the location
atlas_query_method = "peak"

# candiate p_values
candidate_p_values = [0.01, ]

# cluster constraints
cluster_size = 40
criteria_n_cluster = 9
NN_level = 3
thresholds = None

# ROIs
roi_vtk_files = sorted(glob.glob(os.path.join(brain_roi_path, "*.vtk")))
roi_names = [Path(file).stem for file in roi_vtk_files]

In [11]:
brain_roi_path

'/Users/seojin/Desktop/DP_visualization/brain_rois'

# Sort roi vtks

In [12]:
def cmp_roi_name(roi_name1, roi_name2):
    """
    Compare roi name for sorting name list
    
    :param roi_name1: name of roi 1(string)
    :param roi_name2: name of roi 2(string)
    
    return (boolean)
    """
    sp_roi_name1 = roi_name1.split("_")
    sp_roi_name2 = roi_name2.split("_")
    
    roi_name1_ = "_".join(sp_roi_name1[1:]) if len(sp_roi_name1) > 1 else sp_roi_name1[0]
    roi_name2_ = "_".join(sp_roi_name2[1:]) if len(sp_roi_name2) > 1 else sp_roi_name2[0]
    
    return roi_name1_ < roi_name2_

# Sort roi names
roi_vtk_files = sort_usingRef(roi_vtk_files, roi_names , cmp_roi_name)
roi_names = [Path(file).stem for file in roi_vtk_files]

# Get roi name with no orientation
roi_name_noOrients = []
for roi_name in roi_names:
    sp_roi_name = roi_name.split("_")
    
    if len(sp_roi_name) > 1:
        roi_name_noOrient = "_".join(sp_roi_name[1:])
    else:
        roi_name_noOrient = sp_roi_name[0]
    roi_name_noOrients.append(roi_name_noOrient)

# Take colormap
target_rois = ["precentral", "postcentral", "superior_parietal", "inferior_parietal"]
colors = plt.cm.tab10(np.linspace(0, 1, len(target_rois)))
colors = colors[::-1]

# Take color for mapping rois
color_i = 0
prev_roi_name = None
roi_colors = []
for roi_name in roi_name_noOrients:
    if prev_roi_name != None and prev_roi_name != roi_name:
        color_i += 1

    try:
        roi_index = target_rois.index(roi_name)
        roi_colors.append(colors[roi_index])
    except:
        roi_colors.append(hex_to_rgb("#929591") / 255) # gray
    
    prev_roi_name = roi_name

# color -> hex
roi_colors = [to_hex(color) for color in roi_colors]

# Model

In [13]:
stat_map_paths = ["/Users/seojin/Desktop/DP_visualization/statmap/clustSim.nii"]

In [14]:
# 기본적으로 Gray 주고
# 특정 ROI만 색상 지정 (M1, S1, Superior Parietal cortex, Inferior Parietal cortex)

In [17]:
os.path.exists(base_brain_nii_path)

True

In [19]:
os.path.exists(cluster_dir_path)

False

In [15]:
show_clusterize_brain(
    stat_map_paths = stat_map_paths,
    cluster_dir_path = cluster_dir_path,
    base_brain_nii_path = base_brain_nii_path,
    roi_vtk_files = roi_vtk_files,
    thresholds = [2.5758],
    cluster_plot_style = cluster_plot_style,
    atlas_query_method = atlas_query_method,
    cluster_size = cluster_size,
    NN_level = 3,
    cluster_map_colors = f"#ffff00",
    roi_style_info = {
        "opacities" : 0.85,
        "colors" : roi_colors,
        "adjust_methods": [("smooth", {})]
    }
)

base brain vtk path:  /Users/seojin/Desktop/DP_visualization/brain_rois/clusters/group_mask.nii.vtk


[vedo.file_io:267] ERROR: in load(), cannot load /Users/seojin/Desktop/DP_visualization/brain_rois/clusters/group_mask.nii.vtk


AttributeError: 'NoneType' object has no attribute 'opacity'

In [54]:
import vedo

In [56]:
a = vedo.load("/Users/seojin/Desktop/MRI_visualization/Vedo_visualization/group_mask.vtk")

In [58]:
a.opacity

<bound method PointsVisual.opacity of <vedo.mesh.Mesh object at 0x1598f3f90>>